In [2]:
# CELL 1 — LOAD DATA FOR INSIGHTS

import pandas as pd
import os

BASE_PATH      = r"C:\Users\User\Desktop\Data Analysis\Nairobi Public Health Facility Access Analysis"
PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")

master = pd.read_csv(os.path.join(PROCESSED_PATH, "nairobi_master.csv"))

print("Loaded. Shape:", master.shape)
print(master[['constituency','HAII_normalised','rank','facility_density',
              'total_pop','total_facilities','confidence']].to_string())

Loaded. Shape: (17, 20)
        constituency  HAII_normalised  rank  facility_density  total_pop  total_facilities confidence
0            Mathare           100.00     1            0.7262     206564                15       high
1      Embakasi West            36.55     2            0.9900     272733                27        low
2     Embakasi South            32.43     3            0.8807     272519                24        low
3     Embakasi North            31.74     4            1.4334     181388                26        low
4            Ruaraka            25.41     5            1.3498     192620                26       high
5          Kamukunji            19.35     6            1.2984     261855                34     medium
6           Makadara            13.75     7            1.9667     218641                43     medium
7           Kasarani            13.71     8            0.6149     780656                48       high
8   Embakasi Central            12.70     9            2.2

In [3]:
# CELL 2 — KEY STATISTICAL FINDINGS

print("=" * 60)
print("FINDING 1 — MATHARE IS THE MOST UNDERSERVED CONSTITUENCY")
print("=" * 60)
mathare = master[master['constituency'] == 'Mathare'].iloc[0]
nairobi_avg_density = master['facility_density'].mean()
print(f"Mathare facility density    : {mathare['facility_density']:.2f} per 10,000")
print(f"Nairobi average             : {nairobi_avg_density:.2f} per 10,000")
print(f"Mathare is                  : {nairobi_avg_density / mathare['facility_density']:.1f}x below average")
print(f"Population density          : {mathare['density_per_sqkm']:,.0f} persons per km²")
print(f"Total facilities            : {mathare['total_facilities']}")
print(f"Data confidence             : {mathare['confidence']}")

print()
print("=" * 60)
print("FINDING 2 — STAREHE PARADOX")
print("=" * 60)
starehe = master[master['constituency'] == 'Starehe'].iloc[0]
print(f"Starehe total facilities    : {starehe['total_facilities']}")
print(f"Starehe facility density    : {starehe['facility_density']:.2f} per 10,000")
print(f"Starehe HAII rank           : {starehe['rank']}")
print(f"Mathare total facilities    : {mathare['total_facilities']}")
print(f"Facility gap                : {starehe['total_facilities'] - mathare['total_facilities']} more facilities in Starehe")

print()
print("=" * 60)
print("FINDING 3 — SCALE OF UNDERPROVISION")
print("=" * 60)
below_2 = master[master['facility_density'] < 2]
print(f"Constituencies below 2 facilities per 10,000: {len(below_2)} of 17")
print(f"Combined population of underserved areas    : {below_2['total_pop'].sum():,}")
print(f"As % of Nairobi population                  : {below_2['total_pop'].sum() / master['total_pop'].sum() * 100:.1f}%")

print()
print("=" * 60)
print("FINDING 4 — KASARANI ANOMALY")
print("=" * 60)
kasarani = master[master['constituency'] == 'Kasarani'].iloc[0]
print(f"Kasarani population         : {kasarani['total_pop']:,}")
print(f"Kasarani facilities         : {kasarani['total_facilities']}")
print(f"Kasarani facility density   : {kasarani['facility_density']:.2f} per 10,000")
print(f"Kasarani HAII rank          : {kasarani['rank']}")
print(f"NOTE: Large area (136 km²) deflates HAII despite critical facility shortage")

print()
print("=" * 60)
print("FINDING 5 — HOSPITAL-LEVEL CARE CONCENTRATION")
print("=" * 60)
level4_plus = master[['constituency','level_4_count','level_5_count',
                       'level_6_count']].copy()
level4_plus['hospital_count'] = (level4_plus['level_4_count'] +
                                  level4_plus['level_5_count'] +
                                  level4_plus['level_6_count'])
no_hospital = level4_plus[level4_plus['hospital_count'] == 0]
print(f"Constituencies with zero Level 4+ hospitals : {len(no_hospital)}")
print(no_hospital[['constituency','hospital_count']].to_string())
print()
total_hospitals = level4_plus['hospital_count'].sum()
top5_hospitals = level4_plus.nlargest(5, 'hospital_count')['hospital_count'].sum()
print(f"Total Level 4+ hospitals in Nairobi         : {total_hospitals}")
print(f"Held in top 5 constituencies                : {top5_hospitals}")
print(f"As % of all hospitals                       : {top5_hospitals/total_hospitals*100:.1f}%")

FINDING 1 — MATHARE IS THE MOST UNDERSERVED CONSTITUENCY
Mathare facility density    : 0.73 per 10,000
Nairobi average             : 1.96 per 10,000
Mathare is                  : 2.7x below average
Population density          : 68,855 persons per km²
Total facilities            : 15
Data confidence             : high

FINDING 2 — STAREHE PARADOX
Starehe total facilities    : 136
Starehe facility density    : 4.95 per 10,000
Starehe HAII rank           : 12
Mathare total facilities    : 15
Facility gap                : 121 more facilities in Starehe

FINDING 3 — SCALE OF UNDERPROVISION
Constituencies below 2 facilities per 10,000: 11 of 17
Combined population of underserved areas    : 3,133,174
As % of Nairobi population                  : 71.2%

FINDING 4 — KASARANI ANOMALY
Kasarani population         : 780,656
Kasarani facilities         : 48
Kasarani facility density   : 0.61 per 10,000
Kasarani HAII rank          : 8
NOTE: Large area (136 km²) deflates HAII despite critical facility

In [4]:
# CELL 3 — POLICY RECOMMENDATIONS

print("""
POLICY RECOMMENDATIONS
Nairobi Healthcare Access Inequality Analysis
Based on 2019 KNBS Population Data and 2017 KMHFL Facility Data
================================================================

RECOMMENDATION 1 — IMMEDIATE PRIORITY: MATHARE
Mathare has 0.73 facilities per 10,000 people against a Nairobi
average of 1.96. With a population density of 68,855 persons per km²
and only 15 operational facilities, it is the most acutely underserved
constituency in Nairobi by HAII score (100/100, high confidence).
The Nairobi County Government should prioritise construction of at
least 2 new Level 3 health centres in Mathare as an immediate
intervention. This would raise facility density to approximately
1.70 per 10,000 — still below average but closing the gap.

RECOMMENDATION 2 — URGENT: KASARANI HOSPITAL GAP
Kasarani is home to 780,656 people — the largest constituency
population in Nairobi — yet has zero Level 4+ hospitals and only
0.61 facilities per 10,000 people. Residents requiring hospital-level
care must travel to neighbouring constituencies, creating pressure on
facilities in Starehe and Westlands. A single Level 4 county hospital
in Kasarani would serve the largest unmet hospital-access gap in
Nairobi.

RECOMMENDATION 3 — STRUCTURAL: EMBAKASI CLUSTER
Embakasi West, South, and North collectively house approximately
726,640 people (low confidence data) with zero Level 4+ hospitals
across all three constituencies. The cluster shows HAII scores of
31-36, indicating systemic underprovision. A shared Level 4 facility
positioned at the Embakasi cluster boundary would serve all three
constituencies simultaneously and reduce pressure on Makadara and
Kamukunji.

RECOMMENDATION 4 — DECENTRALISATION: HOSPITAL CONCENTRATION
75.7% of all Level 4+ hospitals in Nairobi are held in 5
constituencies. This concentration means peripheral constituencies
are structurally dependent on central facilities. The Nairobi County
Government should adopt a policy of no new hospital construction in
already well-served constituencies until each constituency has at
least one Level 3 facility. New capital should be directed exclusively
to the 6 zero-hospital constituencies: Embakasi South, Embakasi North,
Ruaraka, Kasarani, Embakasi Central, and Dagoretti South.

RECOMMENDATION 5 — DATA: CLOSE THE CONSTITUENCY POPULATION GAP
4 of the top 9 underserved constituencies carry low-confidence
population data sourced from non-KNBS administrative sites. KNBS
should publish a clean constituency-level population table from the
2019 census to enable evidence-based resource allocation at the
constituency level. The current absence of this dataset forces
analysts and policymakers to rely on reconstructed estimates.

================================================================
LIMITATIONS
- Facility data is from 2017 KMHFL; some facilities may have opened
  or closed since then
- Population data for 12 of 17 constituencies is reconstructed from
  mixed sources, not direct KNBS constituency tables
- HAII does not account for facility quality, staffing levels,
  or actual utilisation rates
- Private facility ownership is included in counts but access
  depends on affordability, which this analysis does not measure
================================================================
""")


POLICY RECOMMENDATIONS
Nairobi Healthcare Access Inequality Analysis
Based on 2019 KNBS Population Data and 2017 KMHFL Facility Data

RECOMMENDATION 1 — IMMEDIATE PRIORITY: MATHARE
Mathare has 0.73 facilities per 10,000 people against a Nairobi
average of 1.96. With a population density of 68,855 persons per km²
and only 15 operational facilities, it is the most acutely underserved
constituency in Nairobi by HAII score (100/100, high confidence).
The Nairobi County Government should prioritise construction of at
least 2 new Level 3 health centres in Mathare as an immediate
intervention. This would raise facility density to approximately
1.70 per 10,000 — still below average but closing the gap.

RECOMMENDATION 2 — URGENT: KASARANI HOSPITAL GAP
Kasarani is home to 780,656 people — the largest constituency
population in Nairobi — yet has zero Level 4+ hospitals and only
0.61 facilities per 10,000 people. Residents requiring hospital-level
care must travel to neighbouring constituencies,